In [ ]:
### Resolução do MiniProjeto ###
# Objetivo: Tratar e Analisar os Dados para Alimentar um Dashboard de Vendas
# Descrição: Este código realiza o carregamento, limpeza, análise e visualização de uma base de dados de vendas no varejo, com foco em categorias de produtos e características dos clientes.
#Aluno: Bruno Briani de Paula

#--------- Base de Dados Varejo --------- 
# Origem: Kaggle 
# URL: https://www.kaggle.com/datasets/namespaiva/base-varejo

#--------- Importação das Bibliotecas ---------
import pandas as pd     
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==================================================================
# ETAPA 1: CARREGAR OS DADOS (Tratando separador e Nulos)
# ==================================================================
print("--- 1. CARREGANDO E IDENTIFICANDO NULOS ---")
caminho_arquivo = r"C:\Users\Bruno\Documents\GitHub\Auladehoje\turma-visualizacao-de-dados\alunos\Bruno_Briani\basevarejo.csv"

# sep=';' corrige as colunas e na_values converte o #N/D em nulo real (NaN) na hora da leitura
df = pd.read_csv(caminho_arquivo, sep=';', na_values=["#N/D", "ND", "null", "nan", " ", ""])

print(f"Número de registros original: {df.shape[0]} linhas e {df.shape[1]} colunas.")
print("-" * 60)

In [ ]:
# ==================================================================
# ETAPA 2: LIMPEZA E TRATAMENTO DE DADOS
# ==================================================================

#--- Nesta etapa será feita uma limpeza dos dados para garantir que estejam prontos para análises e visualizações, sem distorções causadas por nulos ou inconsistências. ---
#--- Print inicial para identificar com título a função desta etapa, e prints de confirmação para cada etapa de limpeza aplicada. ---
#--- No dataframe foram identificadas colunas 100% vazias as quais são irrelevantes para as análise que queremos fazer. Para corrigir isso, nomeamos o dataframe (df_limpo), após aplicar a função df.dropna() para eliminar colunas 100% vazias ---
#--- Em seguida, aplicamos a função str.strip() para eliminar espaços em branco invisíveis nos nomes das colunas, garantindo que não haja problemas de acesso às colunas por causa de espaços indesejados. ---
#--- Depois, utilizamos a função drop_duplicates() para eliminar linhas 100% idênticas, mantendo apenas a primeira ocorrência, e printamos o número de linhas duplicadas que foram deletadas. ---
#--- Para a coluna 'DATA', aplicamos a função pd.to_datetime() para converter os valores para o formato de data, especificando o formato 'mixed' para lidar com diferentes formatos de data e usando errors='coerce' para converter valores inválidos em NaT (Not a Time). ---
#--- Em vez de preencher os nulos nas colunas 'PR_CAT' e 'CL_GENERO' com uma string como 'Sem Categoria', optamos por remover as linhas que contêm nulos nessas colunas para evitar distorções nos gráficos e análises. ---
#--- Para garantir que nenhuma string "nan" residual sobreviva, aplicamos uma máscara de texto para filtrar e eliminar quaisquer linhas onde 'PR_CAT' ou 'CL_GENERO' contenham a string "nan" (independente de maiúsculas ou minúsculas). ---
#--- Para a coluna 'CL_FHL' (Número de Filhos), aplicamos uma lógica de imputação onde calculamos a mediana dos valores existentes e preenchemos os nulos com essa mediana, garantindo que a distribuição dos dados não seja distorcida por valores extremos. ---   


print("\n--- 2. APLICANDO ETAPAS DE LIMPEZA ---")

# A. Remove colunas que estejam 100% vazias (colunas sem dados do final)
df_limpo = df.dropna(axis=1, how='all').copy()

# B. Remove espaços em branco invisíveis nos nomes das colunas
df_limpo.columns = df_limpo.columns.str.strip()

# C. Eliminação de duplicadas absolutas (linhas 100% idênticas)
linhas_antes = df_limpo.shape[0]
df_limpo = df_limpo.drop_duplicates(keep='first')
print(f"✔️ Total de linhas duplicadas deletadas: {linhas_antes - df_limpo.shape[0]}")

# D. Ajuste do tipo de dado da coluna DATA
if 'DATA' in df_limpo.columns:
    df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], format='mixed', errors='coerce')
    print("✔️ Coluna 'DATA' convertida para Datetime com sucesso!")

# E. Remoção cirúrgica de linhas com valores nulos ou strings vazias em colunas 
# Em vez de preencher com 'Sem Categoria', removemos para evitar distorções nos gráficos
df_limpo = df_limpo.dropna(subset=['PR_CAT', 'CL_GENERO'])

# Limpeza com máscara de texto para garantir que nenhuma string "nan" residual permaneça na df
df_limpo = df_limpo[
    (df_limpo['PR_CAT'].astype(str).str.strip().str.upper() != 'NAN') & 
    (df_limpo['CL_GENERO'].astype(str).str.strip().str.upper() != 'NAN')
].copy()

# F. Lógica de Imputação para o Número de Filhos (CL_FHL)
if 'CL_FHL' in df_limpo.columns:
    mediana_filhos = df_limpo['CL_FHL'].median()
    df_limpo['CL_FHL'] = df_limpo['CL_FHL'].fillna(mediana_filhos)
    print(f"✔️ Nulos em 'CL_FHL' tratados com a mediana ({mediana_filhos}).")

print(f"Formato final da base limpa para análises: {df_limpo.shape[0]} linhas.")
print("-" * 60)



# ==================================================================
# ETAPA 3: SALVAR RESULTADO FINAL TRATADO
# ==================================================================

#---Função para salvar o dataframe limpo em um novo arquivo CSV, garantindo que as análises futuras sejam feitas com a base de dados tratada e pronta para uso. O arquivo é salvo sem o índice para manter a formatação limpa. ---

df_limpo.to_csv("base_varejo_limpa_final.csv", index=False)
print("\n🎉 Base de dados salva com sucesso como 'base_varejo_limpa_final.csv'!")




--- 2. APLICANDO ETAPAS DE LIMPEZA ---
✔️ Total de linhas duplicadas deletadas: 96553
✔️ Coluna 'DATA' convertida para Datetime com sucesso!
✔️ Nulos em 'CL_FHL' tratados com a mediana (0.0).
Formato final da base limpa para análises: 730219 linhas.
------------------------------------------------------------

🎉 Base de dados salva com sucesso como 'base_varejo_limpa_final.csv'!


In [ ]:
# ==================================================================
# ETAPA 4: ESTATÍSTICA DESCRITIVA
# ==================================================================

#--- Nesta etapa, realizamos uma análise estatística descritiva para entender melhor a distribuição dos dados, identificar tendências e possíveis outliers. ---
#--- A função describe() é utilizada para obter uma visão geral das colunas numéricas, enquanto a inclusão de include=['object', 'string'] permite analisar as colunas categóricas. ---
#--- Para a coluna de número de filhos (CL_FHL), criamos um detalhamento customizado que apresenta métricas como contagem, média, mediana, moda, desvio padrão, mínimo e máximo, oferecendo uma visão mais aprofundada dessa variável específica. ---
#--- Os resultados são apresentados de forma clara e organizada, facilitando a interpretação dos dados para futuras análises e visualizações. ---

print("\n=== ESTATÍSTICA DESCRITIVA DAS COLUNAS NUMÉRICAS ===")
print(df_limpo.describe().round(2))
print("\n" + "="*50 + "\n")

print("=== ESTATÍSTICA DESCRITIVA DAS COLUNAS CATEGÓRICAS ===")
print(df_limpo.describe(include=['object', 'string']))
print("\n" + "="*50 + "\n")

# Detalhamento individual customizado para Filhos (CL_FHL)
if 'CL_FHL' in df_limpo.columns:
    estatisticas_filhos = pd.DataFrame({
        'Métrica': ['Contagem (Registros)', 'Média', 'Mediana', 'Moda', 'Desvio Padrão', 'Mínimo', 'Máximo'],
        'Valor': [
            df_limpo['CL_FHL'].count(),
            df_limpo['CL_FHL'].mean(),
            df_limpo['CL_FHL'].median(),
            df_limpo['CL_FHL'].mode()[0] if not df_limpo['CL_FHL'].mode().empty else None,
            df_limpo['CL_FHL'].std(),
            df_limpo['CL_FHL'].min(),
            df_limpo['CL_FHL'].max()
        ]
    })
    pd.options.display.float_format = '{:.2f}'.format
    print("--- DETALHAMENTO DA COLUNA DE FILHOS (CL_FHL) ---")
    print(estatisticas_filhos.to_string(index=False))
    print("-" * 60)


In [ ]:
# ==================================================================
# ETAPA 5: GERAÇÃO DOS GRÁFICOS DO PRIMEIRO BLOCO (Individuais)
# ==================================================================

#--- Nesta etapa, geramos gráficos individuais para as colunas de interesse, como 'PR_CAT' (Categoria do Produto) e 'CL_GENERO' (Gênero do Cliente), utilizando a biblioteca Seaborn para criar visualizações claras e informativas. ---
#--- Configuramos o tema do Seaborn para 'whitegrid' para melhorar a legibilidade dos gráficos, e ajustamos o tamanho das figuras para garantir que as visualizações sejam claras e fáceis de interpretar. ---  
#--- Os gráficos gerados incluem um countplot horizontal para as categorias de produtos mais vendidos, ordenado pela quantidade, e um countplot vertical para a distribuição de compras por gênero do cliente, ambos utilizando paletas de cores distintas para melhorar a visualização. ---
#--- Foi realizada  o ajuste de configurações de estilo e tamanho para os gráficos ---

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

if 'PR_CAT' in df_limpo.columns:
    plt.figure()
    ordem_categorias = df_limpo['PR_CAT'].value_counts().index
    sns.countplot(data=df_limpo, y='PR_CAT', order=ordem_categorias, hue='PR_CAT', palette='viridis', legend=False)
    plt.title('Top Categorias de Produtos Mais Vendidos', fontsize=14, fontweight='bold')
    plt.xlabel('Quantidade de Itens Vendidos', fontsize=12)
    plt.ylabel('Categoria do Produto Adquirido', fontsize=12)
    plt.tight_layout()
    plt.show()

if 'CL_GENERO' in df_limpo.columns:
    plt.figure()
    sns.countplot(data=df_limpo, x='CL_GENERO', hue='CL_GENERO', palette='pastel', legend=False)
    plt.title('Distribuição de Compras por Gênero do Cliente', fontsize=14, fontweight='bold')
    plt.xlabel('Gênero (CL_GENERO)', fontsize=12)
    plt.ylabel('Total de Registros de Venda', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ==================================================================
# ETAPA 6: DASHBOARD TRIPLICADO (Subplots Unificados)
# ==================================================================
#--- Nesta etapa, criamos um dashboard unificado com três subplots para visualizar simultaneamente a distribuição de compras por gênero, as categorias de produtos mais vendidas e a evolução temporal das vendas. ---
#--- Utilizamos apenas o df_limpo tratado! ---

plt.figure(figsize=(18, 5))

# Subplot 1: Gênero
plt.subplot(1, 3, 1)
sns.countplot(x='CL_GENERO', data=df_limpo, palette='Dark2', hue='CL_GENERO', legend=False)
plt.title('Volume de Compras por Gênero (CL_GENERO)', fontsize=12, fontweight='bold')
plt.xlabel('Gênero')
plt.ylabel('Quantidade de Registros')

# Subplot 2: Categorias
plt.subplot(1, 3, 2)
top_categorias = df_limpo['PR_CAT'].value_counts().head(5)
sns.barplot(x=top_categorias.values, y=top_categorias.index, palette='Blues_r', hue=top_categorias.index, legend=False)
plt.title('Top 5 Categorias Mais Vendidas (PR_CAT)', fontsize=12, fontweight='bold')
plt.xlabel('Quantidade de Itens Vendidos')
plt.ylabel('Categoria')

# Subplot 3: Evolução Temporal
plt.subplot(1, 3, 3)
vendas_por_dia = df_limpo.groupby('DATA').size().sort_index()
vendas_por_dia.plot(kind='line', color='crimson', linewidth=1.5)
plt.title('Evolução Diária de Vendas (Linha do Tempo)', fontsize=12, fontweight='bold')
plt.xlabel('Período')
plt.ylabel('Total de Vendas no Dia')
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# ETAPA 7: TABELAS DINÂMICAS E AGRUPAMENTOS DE PADRÕES
# ==================================================================
#--- Nesta etapa, criamos uma tabela dinâmica para analisar a relação entre categorias de produtos, gênero e estado civil dos clientes, utilizando a função pivot_table do Pandas. ---
#--- A tabela dinâmica é configurada para contar o número de registros (CO_ID) para cada combinação de categoria de produto (PR_CAT), gênero (CL_GENERO) e estado civil (CL_EC), preenchendo os valores nulos com zero para facilitar a interpretação. ---
#--- Em seguida, realizamos um agrupamento multidimensional utilizando a função groupby para identificar os padrões mais comuns de compra considerando as três variáveis (gênero, estado civil e categoria do produto), e apresentamos os resultados ordenados pelo total de compras. ---   
#--- A análise de padrões de compra considerando múltiplas variáveis é fundamental para entender o comportamento do consumidor e identificar segmentos de mercado, permitindo que estratégias de marketing e vendas sejam mais eficazes e direcionadas. --- 


tabela_dinamica = pd.pivot_table(
    df_limpo, 
    values='CO_ID', 
    index=['PR_CAT'],            
    columns=['CL_GENERO', 'CL_EC'], 
    aggfunc='count',             
    fill_value=0                 
)

print("\n=== TABELA DINÂMICA: PRODUTO vs GÊNERO vs ESTADO CIVIL ===")
try:
    from IPython.display import display
    display(tabela_dinamica)
except ImportError:
    print(tabela_dinamica)

# Agrupamento multidimensional (3 Variáveis)
agrupamento_3v = df_limpo.groupby(['CL_GENERO', 'CL_EC', 'PR_CAT']).size().reset_index(name='TOTAL_COMPRAS')
agrupamento_3v = agrupamento_3v.sort_values(by='TOTAL_COMPRAS', ascending=False)

print("\n=== TOP PADRÕES DE AGRUPAMENTO (3 VARIÁVEIS) ===")
print(agrupamento_3v.head(15).to_string(index=False))


In [ ]:
# ==================================================================
# ETAPA 8: GRÁFICO AVANÇADO DE PADRÃO DE CONSUMO MULTIVARIÁVEL
# ==================================================================
#--- Nesta etapa, criamos um gráfico avançado para visualizar os padrões de consumo considerando múltiplas variáveis, especificamente a relação entre categorias de produtos (PR_CAT) e gênero dos clientes (CL_GENERO). ---
#--- Utilizamos um countplot para mostrar a distribuição de compras por categoria de produto, segmentada por gênero, permitindo identificar quais categorias são mais populares entre diferentes gêneros. ---
#--- Este gráfico é fundamental para entender os padrões de consumo considerando múltiplas variáveis, permitindo identificar quais categorias de produtos são mais populares entre diferentes gêneros, o que pode orientar estratégias de marketing e vendas mais eficazes. ---
#--- Utilizamos o df_limpo tratado para garantir que os padrões de consumo sejam analisados com dados limpos e confiáveis, evitando distorções causadas por nulos ou inconsistências. ---
#--- O gráfico gerado é um countplot horizontal que mostra a quantidade de compras para cada categoria de produto (PR_CAT), segmentada por gênero (CL_GENERO), permitindo uma análise visual clara dos padrões de consumo entre diferentes gêneros para cada categoria de produto. ---
#--- Para garantir que o gráfico seja claro e legível, ajustamos o tamanho da figura para (14, 7), configuramos títulos e rótulos com fontes maiores e mais legíveis, e utilizamos uma paleta de cores distinta para diferenciar os gêneros, facilitando a interpretação dos padrões de consumo. ---

plt.figure(figsize=(14, 7))

sns.countplot(
    data=df_limpo, 
    y='PR_CAT', 
    hue='CL_GENERO', 
    order=df_limpo['PR_CAT'].value_counts().index,
    palette='Set2'
)

plt.title('Padrão de Consumo: Categorias de Produto por Gênero do Cliente', fontsize=14, fontweight='bold')
plt.xlabel('Quantidade de Compras', fontsize=12)
plt.ylabel('Categoria do Produto (PR_CAT)', fontsize=12)
plt.legend(title='Gênero (CL_GENERO)')
plt.tight_layout()
plt.show()